### Codice di analisi dei dati generati

Importiamo sia i dati simulati, che quelli generati

In [ ]:
import pandas as pd

def load_primary_tracks(file_path, chunksize=100_000):
    primary_rows = []

    # Lettura a chunk per gestire file grandi
    for chunk in pd.read_csv(file_path, sep="\t", chunksize=chunksize):
        # Filtro dei primari (ParentID == 0)
        primaries = chunk[chunk["ParentID"] == 0]
        primary_rows.append(primaries)

    # Concatenazione finale
    df_primaries = pd.concat(primary_rows, ignore_index=True)
    return df_primaries

# Esempio d'uso
file_path_simulated = "../DatiPerNN_BigMixture_1500keV_PulseTube_1e4EInt.txt"  
df_simulated = load_primary_tracks(file_path_simulated)

file_path_generated = "DatiPerNN_BigMixture_1500keV_PulseTube_1e4EInt_generated.txt"  
df_generated = load_primary_tracks(file_path_generated)

# Anteprima
print("SIMULATED DATA:")
print(df_simulated.head())
print("\n")
print("GENERATED DATA:")
print(df_generated.head())

SIMULATED DATA:
   EventNumber  ParentID  TrackID ParticleName  ParticleEn(MeV)  Edep(MeV)  \
0            0         0        1           e-              1.5   0.581270   
1            0         0        1           e-              1.5   0.396748   
2            0         0        1           e-              1.5   0.091727   
3            0         0        1           e-              1.5   0.052685   
4            0         0        1           e-              1.5   0.058514   

      X(mm)     Y(mm)     Z(mm)  Step(mm)  Track(mm)     VolumeName  \
0 -0.004809  0.182232  0.060429  0.198935   0.198935  crystal_phys0   
1 -0.027830  0.335070  0.121910  0.292686   0.491621  crystal_phys0   
2 -0.007447  0.399106  0.152664  0.142127   0.633748  crystal_phys0   
3 -0.046494  0.394904  0.190319  0.108555   0.742303  crystal_phys0   
4 -0.012974  0.371026  0.204682  0.089589   0.831892  crystal_phys0   

  MotherVolumeName  CenterX(mm)  CenterY(mm)  CenterZ(mm)  
0           Module          

Istogramma dell'energia totale depositata da ogni particella

In [23]:
import plotly.graph_objects as go
import numpy as np

total_energy_simulated = df_simulated.groupby('EventNumber')['Edep(MeV)'].sum()
#total_energy_generated = df_generated.groupby('EventNumber')['Edep(MeV)'].sum()
total_energy_generated = (df_generated.groupby("EventNumber").apply(lambda g: g["Edep(MeV)"].iloc[:-1].sum())) #esclude l'energia depositata nell'ultimo step

counts_simulated, bins_simulated = np.histogram(total_energy_simulated, 300)#, range=(0, 6))
bins_centre_simulated = (bins_simulated[:-1] + bins_simulated[1:]) / 2

counts_generated, bins_generated = np.histogram(total_energy_generated, 300)#, range=(0, 6))
bins_centre_generated = (bins_generated[:-1] + bins_generated[1:]) / 2

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=bins_centre_simulated,
    y=counts_simulated,
    mode='lines',
    name="Simulated data",
    line=dict(color='red', shape='hvh'),  # shape='hvh' crea l'effetto a step
    opacity=0.5,
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=np.concatenate([bins_centre_simulated, bins_centre_simulated[::-1]]),
    y=np.concatenate([counts_simulated, np.zeros_like(counts_simulated)]),
    fill='toself',
    fillcolor='red',
    line=dict(color='red', shape='hvh'),
    opacity=0.2,
    name='Simulated data',
    showlegend=False
))

fig.add_trace(go.Scatter(
    x=bins_centre_generated,
    y=counts_generated,
    mode='lines',
    name="Generated data",
    line=dict(color='blue', shape='hvh'),
    opacity=0.5,
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=np.concatenate([bins_centre_generated, bins_centre_generated[::-1]]),
    y=np.concatenate([counts_generated, np.zeros_like(counts_generated)]),
    fill='toself',
    fillcolor='blue',
    line=dict(color='blue', shape='hvh'),
    opacity=0.2,
    name='Generated data',
    showlegend=False
))

fig.update_layout(
    xaxis_title="Energy [MeV]",
    yaxis_title="Counts",
    template="plotly_white",
    height=600
)

#fig.update_yaxes(type="log")

fig.show()

/tmp/ipykernel_25777/2970496408.py:6: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Istogramma del numero di step di ogni particella

In [24]:
# Numero di step per evento (sia simulati che generati)
steps_simulated = df_simulated.groupby("EventNumber").size()
#steps_generated = df_generated.groupby("EventNumber").size()
steps_generated = df_generated.groupby("EventNumber").size() - 1  # esclude l'ultimo step

# Istogramma simulato
counts_simulated, bins_simulated = np.histogram(steps_simulated, 100, range = (0,50))
bins_centre_simulated = (bins_simulated[:-1] + bins_simulated[1:]) / 2

# Istogramma generato
counts_generated, bins_generated = np.histogram(steps_generated, 100, range = (0,50))
bins_centre_generated = (bins_generated[:-1] + bins_generated[1:]) / 2

# Figura
fig = go.Figure()

# Simulated data (linea + area rossa)
fig.add_trace(go.Scatter(
    x=bins_centre_simulated,
    y=counts_simulated,
    mode="lines",
    name="Simulated data",
    line=dict(color="red", shape="hvh"),
    opacity=0.5,
    showlegend=True
))
fig.add_trace(go.Scatter(
    x=np.concatenate([bins_centre_simulated, bins_centre_simulated[::-1]]),
    y=np.concatenate([counts_simulated, np.zeros_like(counts_simulated)]),
    fill="toself",
    fillcolor="red",
    line=dict(color="red", shape="hvh"),
    opacity=0.2,
    name="Simulated data",
    showlegend=False
))

# Generated data (linea + area blu)
fig.add_trace(go.Scatter(
    x=bins_centre_generated,
    y=counts_generated,
    mode="lines",
    name="Generated data",
    line=dict(color="blue", shape="hvh"),
    opacity=0.5,
    showlegend=True
))
fig.add_trace(go.Scatter(
    x=np.concatenate([bins_centre_generated, bins_centre_generated[::-1]]),
    y=np.concatenate([counts_generated, np.zeros_like(counts_generated)]),
    fill="toself",
    fillcolor="blue",
    line=dict(color="blue", shape="hvh"),
    opacity=0.2,
    name="Generated data",
    showlegend=False
))

# Layout
fig.update_layout(
    xaxis_title="Step number per event",
    yaxis_title="Counts",
    template="plotly_white",
    height=600
)

# fig.update_yaxes(type="log")  # scommenta se vuoi asse Y logaritmico

fig.show()
